# 甄数下游实验：筛过的示范能不能训出更好的策略

数据：robomimic 的 Square（套螺母）MH 数据，300 条真人遥操作示范，6 名操作员各 50 条，按水平分好、中、差 3 档（MIT 授权）。

比 4 组训练数据，每组 3 个随机种子，策略都用 robomimic 论文的 BC-RNN（低维观测），在仿真里各跑 50 次测成功率：

| 组 | 训练用的示范 |
|---|---|
| all | 全部 300 条 |
| assay | 去掉甄数标出的（关节规则判存疑的） |
| random | 随机去掉同样条数（每个种子抽一次） |
| oracle | 去掉水平差的 100 条（用 robomimic 自带的标签，作参照） |

**事先写下的假设，结果怎样都照实报：**
1. assay 组的成功率不低于 all 组。
2. assay 组的成功率高于 random 组（去掉同样多的示范，挑得准才有用）。

**怎么跑：** 菜单“代码执行程序 → 更改运行时类型”选 T4 GPU，然后从上往下逐格运行。第 6 格是几分钟的调试，跑通再往下。第 7 格会实测速度，估出总时长。结果存到你的 Google Drive，断线后重新从头运行，已经跑完的组会自动跳过。

**实测（2026-10-19）：** 普通 T4 上一次训练约 81 分钟，12 次串行约 16 小时。我们实际是用仓库 `experiments/downstream_robomimic/` 里的脚本，在高内存 T4（8 个 CPU）上 4 个并行跑完的，约 5 小时、约 6 个计算单元，结果在 `results/downstream_robomimic.json`。浏览器标签页或电脑睡着、断开超过约半小时，Colab 会收回运行时，已经跑完的组靠 Drive 里的结果保住。

In [ ]:
# 1. 看显卡，挂载 Google Drive 存结果（会弹出授权，用你自己的账号点允许）
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/assay_downstream'
os.makedirs(RESULTS, exist_ok=True)
print('结果存到', RESULTS)

In [ ]:
# 2. 安装 robosuite 1.5.2、robomimic v0.5.0 和甄数（约 3 至 8 分钟）
# 2026 年 10 月 Colab 是 Python 3.13：robosuite 1.5.2 会装 numpy 1.26.4（要现编译）；MuJoCo 要锁 3.3.7，新版建不起 Square 环境；
# robomimic 锁的 transformers 4.41.2 编译不过，BC-RNN 用不到它，所以 robomimic 不装依赖，只补两个小包
!pip -q install "robosuite==1.5.2" "mujoco==3.3.7" huggingface_hub
!rm -rf /content/robomimic && git clone -q --depth 1 --branch v0.5.0 https://github.com/ARISE-Initiative/robomimic.git /content/robomimic
!pip -q install --no-deps -e /content/robomimic
!pip -q install tensorboardX imageio-ffmpeg
!pip -q install "git+https://github.com/hoongyu/assay.git@5dd7737"
# 如果下一格报 numpy 或 import 相关的错：菜单“代码执行程序 → 重启会话”，再从第 1 格往下运行（第 2 格可以跳过）

In [ ]:
# 3. 下载 Square MH 低维数据（123 MB，Hugging Face：robomimic/robomimic_datasets）
from huggingface_hub import hf_hub_download
TASK = 'square'
SRC = hf_hub_download('robomimic/robomimic_datasets', f'v1.5/{TASK}/mh/low_dim_v15.hdf5', repo_type='dataset',
                      local_dir='/content/data')
print(SRC)

In [ ]:
# 4. 甄数逐条检查（只用关节规则，CPU 上约 1 分钟），按操作员水平统计标出了多少
import json, subprocess, collections
from assay.ingest.robomimic import read_robomimic
subprocess.run(['assay', 'check', SRC, '--json', f'/content/assay_{TASK}.json', '--no-frames'], check=True)
rows = json.load(open(f'/content/assay_{TASK}.json', encoding='utf-8'))
quality = {e.id: e.meta['robomimic_quality'] for e in read_robomimic(SRC).episodes}
flagged = {r['episode_id'].rsplit('/', 1)[1] for r in rows if r['label'] != '合格'}
for q in ['better', 'okay', 'worse']:
    ids = [i for i, v in quality.items() if v == q]
    print(q, sum(i.rsplit('/', 1)[1] in flagged for i in ids), '/', len(ids))
print('标出合计', len(flagged), '条')

In [ ]:
# 5. 复制一份数据，写入 4 组训练用的子集标记（mask/exp_*）
import shutil, h5py, numpy as np
WORK = f'/content/data/{TASK}_mh_assay.hdf5'
shutil.copy(SRC, WORK)
SEEDS = [1, 2, 3]
with h5py.File(WORK, 'a') as f:
    demos = sorted(f['data'].keys(), key=lambda k: int(k.split('_')[1]))
    def put(name, keep):
        if f'mask/{name}' in f:
            del f[f'mask/{name}']
        f.create_dataset(f'mask/{name}', data=np.array(sorted(keep, key=lambda k: int(k.split('_')[1])), dtype='S'))
    put('exp_all', demos)
    put('exp_assay', [d for d in demos if d not in flagged])
    for s in SEEDS:
        rng = np.random.default_rng(s)
        drop = set(rng.choice(demos, size=len(flagged), replace=False))
        put(f'exp_random_s{s}', [d for d in demos if d not in drop])
    put('exp_oracle', [d.decode() for d in f['mask/okay_better'][()]])
    for k in sorted(f['mask'].keys()):
        if k.startswith('exp_'):
            print(k, len(f[f'mask/{k}']))

In [ ]:
# 6. 配置和单组训练。先跑一次调试（几分钟）：能出现 'Success_Rate' 就说明整条链路通了
import importlib.util, re, time, glob
import robomimic.utils.torch_utils as TorchUtils
from robomimic.config import config_factory
from robomimic.scripts.train import train

spec = importlib.util.spec_from_file_location('rm_example', '/content/robomimic/examples/train_bc_rnn.py')
rm_example = importlib.util.module_from_spec(spec); spec.loader.exec_module(rm_example)

def make_config(name, filter_key, seed, epochs, rate, n, horizon=400, steps=100):
    config = config_factory(algo_name='bc')
    config.experiment.name = name
    config.train.data = WORK
    config.train.output_dir = '/content/runs'
    config = rm_example.robosuite_hyperparameters(config)   # robomimic 论文的 BC-RNN 低维超参数
    config.experiment.validate = False
    config.experiment.logging.terminal_output_to_txt = True
    config.experiment.render_video = False
    config.experiment.save.enabled = False
    config.experiment.epoch_every_n_steps = steps
    config.experiment.rollout.rate = rate
    config.experiment.rollout.n = n
    config.experiment.rollout.horizon = horizon
    config.train.hdf5_filter_key = filter_key
    config.train.hdf5_validation_filter_key = None
    config.train.num_epochs = epochs
    config.train.seed = seed
    return config

def run(name, filter_key, seed, epochs, rate, n, horizon=400, steps=100):
    shutil.rmtree(f'/content/runs/{name}', ignore_errors=True)   # 目录已存在时 robomimic 会停下来问，先删掉
    config = make_config(name, filter_key, seed, epochs, rate, n, horizon, steps)
    t0 = time.time()
    train(config, device=TorchUtils.get_torch_device(try_to_use_cuda=True))
    wall = time.time() - t0
    log = open(glob.glob(f'/content/runs/{name}/*/logs/log.txt')[0], encoding='utf-8').read()
    curve = {str(int(e)): float(s) for e, s in re.findall(r'Epoch (\d+) Rollouts took.*?"Success_Rate": ([0-9.]+)', log, re.S)}
    return {'name': name, 'filter_key': filter_key, 'seed': seed, 'epochs': epochs, 'rollouts_per_eval': n,
            'success_by_epoch': curve, 'wall_seconds': round(wall, 1)}

debug = run('debug', 'exp_all', 1, epochs=2, rate=1, n=2, horizon=10, steps=3)
print(debug)

In [ ]:
# 7. 计时：训 10 轮（每轮 100 步），在第 10 轮跑 5 次评估，估出正式实验要多久
probe = run('probe', 'exp_all', 1, epochs=10, rate=10, n=5)
log = open(glob.glob('/content/runs/probe/*/logs/log.txt')[0], encoding='utf-8').read()
EPOCHS, RATE, N = 600, 100, 50          # 正式实验：600 轮，每 100 轮评估一次，每次 50 回合
CONDITIONS = ['exp_all', 'exp_assay', 'exp_random', 'exp_oracle']
per_rollout = float(re.findall(r'Rollouts took ([0-9.]+)s', log)[-1])   # robomimic 打印的是每回合平均秒数
per_epoch = (probe['wall_seconds'] - per_rollout * 5) / 10
one_run = per_epoch * EPOCHS + per_rollout * N * (EPOCHS // RATE)
total_h = one_run * len(CONDITIONS) * len(SEEDS) / 3600
print(f'每轮训练约 {per_epoch:.1f} 秒，每回合评估约 {per_rollout:.1f} 秒（策略还没学会时回合会跑满，偏保守）')
print(f'一组约 {one_run/60:.0f} 分钟，{len(CONDITIONS)} 组 × {len(SEEDS)} 个种子共约 {total_h:.1f} 小时')
print('T4 每小时消耗的计算单元以 Colab 右上角“资源”面板为准；太久的话把 SEEDS 改成 [1, 2] 或 N 改成 25')

In [ ]:
# 8. 正式实验：结果逐组存进 Google Drive，断线后重新运行会跳过已完成的组
for seed in SEEDS:
    for cond in CONDITIONS:
        key = f'{cond}_s{seed}' if cond == 'exp_random' else cond
        name = f'{TASK}_{cond}_s{seed}'
        path = f'{RESULTS}/{name}.json'
        if os.path.exists(path):
            print('跳过（已完成）', name); continue
        res = run(name, key, seed, EPOCHS, RATE, N)
        res['n_demos'] = len(h5py.File(WORK, 'r')[f'mask/{key}'])
        json.dump(res, open(path, 'w'), ensure_ascii=False, indent=1)
        print(name, res['success_by_epoch'], f"{res['wall_seconds']/60:.0f} 分钟")

In [ ]:
# 9. 汇总：最后一次评估和历次最好的成功率（均值 ± 标准差），画学习曲线；把最后打印的 JSON 整段发给项目窗口
import matplotlib.pyplot as plt, statistics as st
allres = [json.load(open(p)) for p in sorted(glob.glob(f'{RESULTS}/{TASK}_exp_*.json'))]
summary = {}
for cond in CONDITIONS:
    rs = [r for r in allres if r['name'].startswith(f'{TASK}_{cond}_')]
    if not rs:
        continue
    last = [r['success_by_epoch'][str(max(map(int, r['success_by_epoch'])))] for r in rs]
    best = [max(r['success_by_epoch'].values()) for r in rs]
    summary[cond] = {'n_demos': [r['n_demos'] for r in rs], 'seeds': [r['seed'] for r in rs],
                     'last_mean': round(st.mean(last), 3), 'last_sd': round(st.pstdev(last), 3),
                     'best_mean': round(st.mean(best), 3), 'best_sd': round(st.pstdev(best), 3),
                     'gpu_minutes': round(sum(r['wall_seconds'] for r in rs) / 60, 1)}
    xs = sorted(int(e) for e in rs[0]['success_by_epoch'])
    ys = [st.mean(r['success_by_epoch'][str(e)] for r in rs) for e in xs]
    plt.plot(xs, ys, marker='o', label=cond.replace('exp_', ''))
plt.xlabel('epoch'); plt.ylabel('success rate'); plt.legend(); plt.title(f'robomimic {TASK} MH, BC-RNN')
plt.savefig(f'{RESULTS}/{TASK}_curves.png', dpi=150); plt.show()
json.dump(summary, open(f'{RESULTS}/{TASK}_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))